# GEO Age-Associated Gene Expression Analysis

NCBI GEOのマイクロアレイおよびbulk RNA-seqデータを用いて、年齢と指定遺伝子の発現量の関連を探索するNotebookです。
`GSE_ID`、`GENE_ID`、年齢区分を設定して実行します。対応するRNA-seq行列が自動取得できない場合は、ローカルファイルを指定してください。

解析: OLS回帰、Spearman相関、Young/Old群のWelchのt検定。結果はCSVとPNGに保存されます。

**注意:** データセットごとの年齢メタデータ・対象試料・発現値の尺度を確認してください。RNA-seqの解析は探索的で、DESeq2等の正式な差次的発現解析の代替ではありません。


## 0. パッケージの導入（初回のみ）
インターネットにつながるPython/Jupyter環境で実行してください。

In [ ]:
%pip install -q GEOparse pandas numpy scipy statsmodels matplotlib requests

## 1. 設定
通常は `GSE_ID` と `GENE_ID` を変更します。GSEごとにRNA-seq発現行列が取得可能とは限りません。その場合はローカルファイルを設定してください。データの品質・研究対象・年齢表記を必ず確認します。
**GSE113957向けの例**：`GSE_ID="GSE113957"`、`CHARACTERISTIC_FILTERS={"disease":["Normal"]}`（健常者のみ）。早老症候群のHGPS群を通常加齢の対照群として混ぜないでください。RNA-seqの発現行列はGSEごとに形式が異なります。GSE113957は投稿者提供のFPKM行列を掲載しており、年齢抽出の修正だけでは`ncbi_tpm`が取得可能とは限りません。

In [ ]:
GSE_ID = "GSE18876"
GENE_ID = 327                       # NCBI Gene ID。文字列/整数どちらでも可
YOUNG_MAX = 30                      # Young: age < 30
OLD_MIN = 60                        # Old: age >= 60

DATA_DIR = "data"
RESULTS_DIR = "results"
TECHNOLOGY = "auto"                 # "auto" / "microarray" / "rna_seq"

# マイクロアレイ専用（GPL注釈列は自動探索）
EXPRESSION_SCALE = "unknown"        # "unknown" / "log2" / "linear"
GENE_COLUMN_OVERRIDES = {}          # 例: {"GPL123": ["ENTREZ_GENE_ID"]}

# RNA-seq専用: NCBIの生成行列、または手元の発現行列
RNA_SEQ_SOURCE = "ncbi_tpm"          # "ncbi_tpm" / "ncbi_raw" / "local_tpm" / "local_raw" / "local_log2"
RNA_SEQ_MATRIX_PATH = None          # local_*を使う場合に指定（CSV/TSV/TXT/.gz）
RNA_SEQ_FEATURE_COLUMN = None       # None: 最初の列 / 指定例: "GeneID" or "gene_id"
RNA_SEQ_GENE_MAP_CSV = None         # 任意: CSVのfeature_id,gene_id列でEnsembl等→NCBI Gene ID変換
RNA_SEQ_SAMPLE_MAP_CSV = None       # 任意: CSVのmatrix_column,gsm_id列で試料列名を対応付け
RNA_SEQ_SEPARATOR = "auto"          # "auto" / "," / "\t" など
RNA_SEQ_PSEUDOCOUNT = 1.0           # log2(TPM + 1) または log2(CPM + 1)
RNA_SEQ_MAX_DOWNLOAD_MB = 1500      # GEOファイル取得上限

AGE_OVERRIDE_CSV = None             # CSVの列: gsm_id,age（年）
CHARACTERISTIC_FILTERS = {}         # 例: {"disease": ["Normal"]}。値は大小文字を無視した完全一致
DROP_MISSING_AGE = False            # 未判定年齢があれば既定で停止
INCLUDE_GSM_IDS = None              # 任意: ["GSM...", ...]
EXCLUDE_GSM_IDS = []
INCLUDE_GPL_IDS = None              # 任意: ["GPL...", ...]
MIN_REGRESSION_N = 5
MIN_GROUP_N = 2
SHOW_PLOTS = True

## 2. GEOメタデータ取得・年齢抽出
`CHARACTERISTIC_FILTERS` は年齢未確定のチェック前に適用されます。除外された試料は `sample_selection_audit.csv` で確認できます。

In [ ]:
from pathlib import Path
import re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import GEOparse
import requests
from urllib.parse import urlencode
from datetime import datetime, timezone
from scipy.stats import spearmanr, ttest_ind
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from IPython.display import display

assert re.fullmatch(r"GSE\d+", str(GSE_ID), flags=re.I), "GSE_IDはGSE12345形式で入力"
assert str(GENE_ID).isdigit(), "GENE_IDは数値のNCBI Gene ID"
assert 0 < YOUNG_MAX < OLD_MIN, "年齢区分の指定を確認"
assert EXPRESSION_SCALE in {"unknown", "log2", "linear"}
assert TECHNOLOGY in {"auto", "microarray", "rna_seq"}
assert RNA_SEQ_SOURCE in {"ncbi_tpm", "ncbi_raw", "local_tpm", "local_raw", "local_log2"}
assert RNA_SEQ_PSEUDOCOUNT > 0

GSE_ID = str(GSE_ID).upper()
GENE_ID = int(GENE_ID)
root = Path(RESULTS_DIR) / f"{GSE_ID}_GeneID_{GENE_ID}"
root.mkdir(parents=True, exist_ok=True)
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

gse = GEOparse.get_GEO(geo=GSE_ID, destdir=DATA_DIR, silent=True)
print("GSE:", GSE_ID, "Samples:", len(gse.gsms), "GPL:", list(gse.gpls))
if not gse.gsms: raise ValueError("このGSEにはGSMが見つかりません")

In [ ]:
AGE_KEYS = re.compile(r"\b(?:age\s*at\s*(?:biopsy|sampling|collection|diagnosis)|donor\s*age|patient\s*age|subject\s*age|chronological\s*age|age)\b", re.I)
# 年・月・週・日が連続する表記（例: 2yr3mos, 6ys11mos）も分割して読む。
AGE_COMPONENT = re.compile(
    r"^(\d{1,3}(?:\.\d+)?)\s*(years?|yrs?|yr|ys|y/o|yo|y|months?|mos?|mo|weeks?|wks?|wk|days?|day|d)(?![A-Za-z])",
    re.I
)
AGE_SCALAR = re.compile(r"^(\d{1,3}(?:\.\d+)?)(?!\w)")

def _years_from_unit(value, unit):
    unit = unit.lower()
    if unit in {'y', 'yr', 'yrs', 'ys', 'yo', 'y/o', 'year', 'years'}:
        return value
    if unit in {'mo', 'mos', 'month', 'months'}:
        return value / 12.0
    if unit in {'wk', 'wks', 'week', 'weeks'}:
        return value / 52.1775
    if unit in {'d', 'day', 'days'}:
        return value / 365.25
    raise ValueError(f'想定外の単位: {unit}')

def _parse_age_value(tail):
    tail = str(tail).strip()
    if not tail or re.match(r'^(?:range|group|category)\b', tail, re.I):
        return np.nan
    # 年齢範囲は平均値で代用しない。
    if re.match(r'^\d+(?:\.\d+)?\s*(?:-|–|—|to)\s*\d+', tail, re.I):
        return np.nan
    # 単位つきは 2yr3mos / 2yr 3mos / 6ys11mos などを連続処理。
    rest = tail
    total = 0.0
    n = 0
    while n < 4:
        m = AGE_COMPONENT.match(rest)
        if not m:
            break
        total += _years_from_unit(float(m.group(1)), m.group(2))
        rest = rest[m.end():].lstrip(' \t,;_+&')
        n += 1
    if n:
        # 残余が別の数値で始まる場合、情報を落としてしまうため未確定扱い。
        if re.match(r'^\d', rest) or re.match(r'^\s*(?:-|–|—|to)\s*\d', rest, re.I):
            return np.nan
        return total if 0 <= total <= 120 else np.nan
    m = AGE_SCALAR.match(tail)
    if m:
        rest = tail[m.end():]
        if re.match(r'^\s*(?:-|–|—|to)\s*\d', rest, re.I):
            return np.nan
        age = float(m.group(1))
        return age if 0 <= age <= 120 else np.nan
    return np.nan

def parse_age_from_text(s):
    """年齢ラベルのある情報のみ読取。曖昧な年齢範囲は推定しない。"""
    s = str(s).strip()
    for m in AGE_KEYS.finditer(s):
        tail = s[m.end():]
        tail = re.sub(r"^\s*\((?:years?|yrs?|months?)\)", "", tail, flags=re.I).lstrip(' :;=,-_')
        age = _parse_age_value(tail)
        if pd.notna(age):
            return age
    return np.nan

def extract_characteristic(gsm, field):
    """GEO characteristics_ch1の 'disease: Normal' 等から指定フィールドを取得。"""
    field = str(field).casefold().strip()
    for key, vals in gsm.metadata.items():
        if not key.startswith('characteristics'):
            continue
        for v in (vals if isinstance(vals, list) else [vals]):
            left, sep, right = str(v).partition(':')
            if sep and left.strip().casefold() == field:
                return right.strip()
    return None

def extract_age(gsm):
    hits=[]
    for key, val in gsm.metadata.items():
        if key in {'characteristics_ch1','title','description','source_name_ch1'} or 'age' in key.lower():
            for s in (val if isinstance(val,list) else [val]):
                age = parse_age_from_text(s)
                if pd.notna(age): hits.append((float(age), key, str(s)))
    unique = {round(x[0],4) for x in hits}
    if len(unique)>1:
        return np.nan, 'CONFLICT', '; '.join(x[2] for x in hits)
    return (hits[0][0], hits[0][1], hits[0][2]) if hits else (np.nan, 'NOT_FOUND','')

rows=[]
for gsm_id, gsm in gse.gsms.items():
    a, source, raw = extract_age(gsm)
    platform = (gsm.metadata.get('platform_id') or [''])[0]
    rows.append(dict(gsm_id=gsm_id, platform_id=platform, age=a, age_source=source,
                     age_raw=raw, title='; '.join(map(str, gsm.metadata.get('title', []))),
                     disease=extract_characteristic(gsm, 'disease')))
metadata = pd.DataFrame(rows)
# 対象外を含むGSE全サンプルの監査情報を保存。
metadata.to_csv(root/'metadata_age_audit.csv', index=False,encoding='utf-8-sig')
if AGE_OVERRIDE_CSV:
    overrides=pd.read_csv(AGE_OVERRIDE_CSV,dtype={'gsm_id':str})
    if not {'gsm_id','age'}.issubset(overrides.columns): raise ValueError('年齢CSVにはgsm_id,age列が必要')
    if overrides.gsm_id.duplicated().any(): raise ValueError('年齢CSVのgsm_idが重複')
    overrides['age']=pd.to_numeric(overrides.age,errors='coerce')
    if overrides.age.dropna().lt(0).any() or overrides.age.dropna().gt(120).any(): raise ValueError('年齢が0～120の範囲外')
    ov=overrides.set_index('gsm_id')['age']
    use=metadata.gsm_id.isin(ov.index) & metadata.gsm_id.map(ov).notna()
    metadata.loc[use,'age']=metadata.loc[use,'gsm_id'].map(ov)
    metadata.loc[use,'age_source']='OVERRIDE_CSV'

selection = metadata[['gsm_id','platform_id','title','disease','age','age_source']].copy()
selection['included'] = True
selection['exclusion_reason'] = ''

def _exclude(mask, label):
    global metadata, selection
    exclude_ids=set(metadata.loc[~mask, 'gsm_id'])
    if exclude_ids:
        sel = selection.gsm_id.isin(exclude_ids)
        selection.loc[sel, 'included'] = False
        selection.loc[sel, 'exclusion_reason'] = label
    metadata = metadata.loc[mask].copy()

if INCLUDE_GSM_IDS is not None:
    _exclude(metadata.gsm_id.isin(INCLUDE_GSM_IDS), 'NOT_IN_INCLUDED_GSMS')
_exclude(~metadata.gsm_id.isin(EXCLUDE_GSM_IDS), 'EXCLUDED_GSM')
if INCLUDE_GPL_IDS is not None:
    _exclude(metadata.platform_id.isin(INCLUDE_GPL_IDS), 'GPL_NOT_INCLUDED')

for field, accepted in CHARACTERISTIC_FILTERS.items():
    accepted = [accepted] if isinstance(accepted,str) else list(accepted)
    accepted_normalized={str(x).strip().casefold() for x in accepted}
    actual = metadata.gsm_id.map(lambda sid: extract_characteristic(gse.gsms[sid],field))
    passed = actual.fillna('').str.strip().str.casefold().isin(accepted_normalized)
    print(f"特性フィルター {field}: {int(passed.sum())}/{len(passed)} 件を採用")
    _exclude(passed, f'CHARACTERISTIC_FILTER_{field}')
selection.to_csv(root/'sample_selection_audit.csv',index=False,encoding='utf-8-sig')
if metadata.empty:
    raise ValueError('選別後のGSMが0件です。CHARACTERISTIC_FILTERSの値をGEO Characteristicsで確認してください。')
missing=metadata.age.isna()
if missing.any():
    print('年齢が未確定のサンプル:', int(missing.sum()))
    display(metadata.loc[missing,['gsm_id','platform_id','age_source','disease','title']].head(20))
    if not DROP_MISSING_AGE:
        raise ValueError(f'年齢を確定できません。{root}/metadata_age_audit.csvとsample_selection_audit.csvを確認してください。')
    metadata=metadata.loc[~missing].copy()
if metadata.empty:
    raise ValueError('年齢が確定したサンプルはありません')
metadata['age_group']=np.select([metadata.age<YOUNG_MAX,metadata.age>=OLD_MIN],['Young','Old'],default='Middle')
metadata.to_csv(root/'metadata_used.csv',index=False,encoding='utf-8-sig')
print('有効GSM:',len(metadata)); display(metadata.groupby(['platform_id','age_group']).size().rename('n').to_frame())


## 3. アッセイ（測定技術）の判定
GSE/GSMの `type`、`library_strategy` 等を参照します。マイクロアレイとRNA-seqが混在している場合、誤ったデータ結合を避けるため自動モードでは停止します。明示的に `TECHNOLOGY` を指定し、`INCLUDE_GSM_IDS` 等で対象GSMを絞ってください。

In [ ]:
def classify_assay(metadata_dict):
    texts = []
    for key in ("type", "library_strategy", "library_selection", "technology"):
        texts.extend(map(str, metadata_dict.get(key, [])))
    val = " ".join(texts).lower()
    if any(t in val for t in ("expression profiling by high throughput sequencing", "rna-seq", "rna seq", "transcriptomic")):
        return "rna_seq"
    if any(t in val for t in ("expression profiling by array", "microarray", "array")):
        return "microarray"
    return "unknown"

sample_assays = []
for gsm_id in metadata.gsm_id:
    gsm = gse.gsms[gsm_id]
    label = classify_assay(gsm.metadata)
    sample_assays.append(label if label != 'unknown' else classify_assay(gse.metadata))
metadata['assay_detected'] = sample_assays
metadata.to_csv(root/'metadata_used.csv',index=False,encoding='utf-8-sig')
print('技術判定の内訳:'); display(metadata.assay_detected.value_counts().rename('n').to_frame())
known = set(metadata.assay_detected) - {'unknown'}
if TECHNOLOGY == 'auto':
    if len(known) != 1:
        raise ValueError('GEOから単一の実験技術を確定できません。metadata_used.csvを確認しTECHNOLOGYを指定してください。')
    assay = next(iter(known))
else:
    assay = TECHNOLOGY
    mismatch = metadata.assay_detected.isin({'microarray','rna_seq'}) & (metadata.assay_detected != assay)
    if mismatch.any():
        raise ValueError('指定TECHNOLOGYとGSMの検出技術が一致しない試料が含まれます。INCLUDE_GSM_IDS等で絞ってください。')
print('解析モード:', assay)

## 4. マイクロアレイ：Gene ID → プローブ照合
GPLの候補注釈列を全て照合し、各プローブを独立解析します。RNA-seqの場合はスキップします。

In [ ]:
# GPLの候補列を複数探索し、Gene IDの完全一致で対応プローブを特定する。
# 注: 遺伝子シンボルやRefSeq accessionの数値をGene IDと混同しない。
GENE_ID_ALIASES = ['gene_assignment','entrez_gene_id','entrez gene id','entrez_gene',
    'gene id','gene_id','geneid','ncbi_gene_id','ncbi gene id','entrez id',
    'entrezgene','entrezgeneid','geneid (entrez)','entrez_gene_ids']

def normalize_col(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def candidate_gene_columns(gpl_id, ann):
    if gpl_id in GENE_COLUMN_OVERRIDES:
        override = GENE_COLUMN_OVERRIDES[gpl_id]
        requested = [override] if isinstance(override, str) else list(override)
        missing = [c for c in requested if c not in ann.columns]
        if missing:
            raise ValueError(f'{gpl_id}: 指定注釈列が存在しません: {missing}')
        return requested
    alias_keys = {normalize_col(a) for a in GENE_ID_ALIASES}
    return [c for c in ann.columns if normalize_col(c) in alias_keys]

def match_gene(v, gene_id, col):
    if pd.isna(v):
        return False
    value = str(v).strip()
    target = str(int(gene_id))
    if normalize_col(col) == 'geneassignment':
        for record in re.split(r'\s*///\s*', value):
            parts = [p.strip() for p in record.split('//')]
            # GPL5175等のgene_assignment: 5番目の要素がEntrez Gene ID
            if len(parts) >= 5 and target in re.findall(r'(?<!\d)\d+(?!\d)', parts[4]):
                return True
        return False
    # 非数値の複合注釈からIDを誤認しないよう、区切り文字に分離された整数だけ許容
    tokens = re.split(r'\s*(?:///|[;,|])\s*', value)
    return target in [t.strip() for t in tokens]

In [ ]:
def build_microarray_mapping():
    mapping = []
    annotation_report = []
    for gpl_id in metadata.platform_id.dropna().unique():
        gpl = gse.gpls.get(gpl_id)
        if gpl is None:
            annotation_report.append(dict(gpl_id=gpl_id,status='GPL_NOT_LOADED',column='',n=0,
                                          available_columns='',comment='GPLオブジェクトが見つかりません'))
            continue
        a = gpl.table
        columns = candidate_gene_columns(gpl_id, a)
        if 'ID' not in a.columns or not columns:
            annotation_report.append(dict(gpl_id=gpl_id,status='ANNOTATION_COLUMN_NOT_FOUND',column='',n=0,
                                          available_columns=' | '.join(map(str,a.columns)),
                                          comment='GPLの注釈列を確認してGENE_COLUMN_OVERRIDESを設定'))
            print(f'{gpl_id} 注釈列候補:',list(a.columns))
            continue
        matches_by_col = {}
        for col in columns:
            mask = a[col].map(lambda v: match_gene(v, GENE_ID, col))
            ids = set(a.loc[mask, 'ID'].dropna().astype(str))
            matches_by_col[col] = ids
            annotation_report.append(dict(gpl_id=gpl_id,
                                          status='OK' if ids else 'GENE_NOT_FOUND',
                                          column=col,n=len(ids),
                                          available_columns=' | '.join(map(str,a.columns)),
                                          comment='列別の照合結果'))
        # 異なる注釈列間の不整合があれば監査対象として警告
        nonempty = [s for s in matches_by_col.values() if s]
        if len(nonempty)>1 and any(s != nonempty[0] for s in nonempty[1:]):
            warnings.warn(f'{gpl_id}: Gene ID対応プローブが注釈列間で一致しません。annotation_audit.csvを確認してください')
        union = set().union(*matches_by_col.values())
        for probe_id in sorted(union):
            sources = [c for c, ids in matches_by_col.items() if probe_id in ids]
            mapping.append(dict(platform_id=gpl_id, probe_id=probe_id,
                                annotation_column=' | '.join(sources)))
    annotation_report = pd.DataFrame(annotation_report)
    annotation_report.to_csv(root/'annotation_audit.csv',index=False,encoding='utf-8-sig')
    print('注釈の検査:')
    display(annotation_report[['gpl_id','status','column','n','comment']])
    if not mapping:
        raise ValueError(
            f'Gene ID {GENE_ID} に対応するプローブが見つかりません。'
            f' {root / "annotation_audit.csv"} のstatusとavailable_columnsを確認してください。'
            ' GPL側にGene ID注釈がない場合は、公式のGPL注釈ファイルなどで対応表を確認する必要があります。'
        )
    mapping = pd.DataFrame(mapping)
    mapping.to_csv(root/'probe_mapping.csv',index=False,encoding='utf-8-sig')
    display(mapping.head(20))
    return mapping

## 5. RNA-seq：行列の取得・Gene ID検索

NCBI生成RNA-seq行列は、GEOが配布する遺伝子ID×GSMのTSV.gzを利用します。**ヒトのGRCh38.p13**の公開ファイル名を用いて直接取得します。非公開/未生成/参照ゲノム形式が異なる場合は取得に失敗します。その場合、`RNA_SEQ_SOURCE="local_tpm"` 等と `RNA_SEQ_MATRIX_PATH` を指定してください。

ローカルファイルの**各行はfeature（gene）、各列はGSM相当の試料**を想定します。Ensembl IDやSymbolだけの行列は自動的にNCBI Gene IDへ変換しません。必要なら`RNA_SEQ_GENE_MAP_CSV`で明示します。

In [ ]:
def geo_rnaseq_filename(gse_id, source):
    suffix = {'ncbi_tpm': 'norm_counts_TPM_GRCh38.p13_NCBI.tsv.gz',
              'ncbi_raw': 'raw_counts_GRCh38.p13_NCBI.tsv.gz'}[source]
    return f'{gse_id}_{suffix}'

def download_ncbi_rnaseq(gse_id, source):
    organisms = set()
    for gsm_id in metadata.gsm_id:
        organisms.update(map(str, gse.gsms[gsm_id].metadata.get('organism_ch1', [])))
    if organisms != {'Homo sapiens'}:
        raise ValueError('NCBI自動取得は本NotebookではHomo sapiens専用です。organism_ch1の確認後、ローカル行列を指定してください。')
    fname = geo_rnaseq_filename(gse_id, source)
    folder = Path(DATA_DIR) / gse_id
    folder.mkdir(parents=True,exist_ok=True)
    file = folder / fname
    if file.exists():
        with file.open('rb') as f: magic=f.read(2)
        if magic != b'\x1f\x8b':
            raise ValueError(f'キャッシュファイルがgzipではありません: {file}。削除して再実行してください。')
        return file, 'cache'
    params = {'type': 'rnaseq_counts','acc':gse_id,'format':'file','file':fname}
    url = 'https://www.ncbi.nlm.nih.gov/geo/download/?' + urlencode(params)
    part = file.with_name(file.name+'.part')
    limit = int(RNA_SEQ_MAX_DOWNLOAD_MB*1024*1024)
    total=0
    try:
        with requests.get(url,stream=True,timeout=(20,120),headers={'User-Agent':'GeoAgeGeneNotebook/3.0'}) as r:
            r.raise_for_status()
            ct = r.headers.get('content-type','').lower()
            with part.open('wb') as w:
                for buf in r.iter_content(chunk_size=1024*1024):
                    if not buf:continue
                    total+=len(buf)
                    if total>limit:raise ValueError(f'ファイルが上限{RNA_SEQ_MAX_DOWNLOAD_MB}MBを超過')
                    w.write(buf)
        with part.open('rb') as f: magic=f.read(2)
        if magic != b'\x1f\x8b':
            raise ValueError('GEOからgzip形式のRNA-seq行列を取得できませんでした（未生成・URL変更等）。'+
                             f'Content-Type={ct}。GEOのDownload Dataを確認するかローカル行列を指定してください。')
        part.replace(file)
    finally:
        part.unlink(missing_ok=True)
    return file,url

def read_sample_mapping(file_path):
    if not file_path:return {}
    tab=pd.read_csv(file_path, dtype=str)
    if not {'matrix_column','gsm_id'}.issubset(tab.columns):
        raise ValueError('RNA_SEQ_SAMPLE_MAP_CSVにはmatrix_column,gsm_idが必要')
    if tab.matrix_column.duplicated().any() or tab.gsm_id.duplicated().any():
        raise ValueError('サンプル対応表に重複があります。1試料1列にしてください')
    return dict(zip(tab.matrix_column,tab.gsm_id))

def read_gene_mapping(file_path,target_gene_id):
    if not file_path:return None
    tab=pd.read_csv(file_path,dtype=str)
    if not {'feature_id','gene_id'}.issubset(tab.columns):
        raise ValueError('RNA_SEQ_GENE_MAP_CSVにはfeature_id,gene_idが必要')
    tab['gene_id']=tab['gene_id'].str.strip()
    features=tab.loc[tab.gene_id == str(target_gene_id),'feature_id'].astype(str).str.strip()
    if not len(features):
        raise ValueError(f'遺伝子対応表にはGene ID {target_gene_id}がありません')
    return set(features)

def detect_delimiter(path):
    if RNA_SEQ_SEPARATOR != 'auto':return RNA_SEQ_SEPARATOR
    name=path.name.lower()
    return ',' if (name.endswith('.csv') or name.endswith('.csv.gz')) else '\t'

def match_column_to_gsm(name, explicit):
    label=str(name)
    if label in explicit:return explicit[label]
    if re.fullmatch(r'GSM\d+',label,re.I):return label.upper()
    ids=set(re.findall(r'GSM\d+',label,re.I))
    return next(iter(ids)).upper() if len(ids)==1 else None

def extract_rnaseq_target(matrix_path, gene_id, mode, selected_gsms):
    """Read a single target feature but normalize raw counts using all feature rows."""
    matrix_path=Path(matrix_path)
    if not matrix_path.is_file():raise FileNotFoundError(matrix_path)
    sep=detect_delimiter(matrix_path)
    # First line only, preserve feature ID as literal text including leading zeros.
    header=pd.read_csv(matrix_path,sep=sep,nrows=0,encoding='utf-8-sig')
    names=header.columns.tolist()
    if len(names)<2:raise ValueError('RNA-seq行列にはfeature ID列と試料列が必要です')
    feature_col=RNA_SEQ_FEATURE_COLUMN or names[0]
    if feature_col not in names:raise ValueError(f'feature列がありません: {feature_col}。実際の列={names[:20]}')
    explicit=read_sample_mapping(RNA_SEQ_SAMPLE_MAP_CSV)
    sel=set(selected_gsms)
    sample_cols={}
    for n in names:
        if n == feature_col:continue
        gsm_id=match_column_to_gsm(n,explicit)
        if gsm_id in sel:
            if gsm_id in sample_cols:
                raise ValueError(f'{gsm_id}に対応する複数列があります。RNA_SEQ_SAMPLE_MAP_CSVで1対1にしてください')
            sample_cols[gsm_id]=n
    if not sample_cols:
        raise ValueError('行列中にGSM列がありません。列名を確認しRNA_SEQ_SAMPLE_MAP_CSVでmatrix_column,gsm_idを対応させてください。')
    missing=sel-set(sample_cols)
    if missing:warnings.warn(f'RNA-seq行列に存在しないGSMが{len(missing)}件: {sorted(missing)[:10]}（除外します）')
    print('RNA-seq対応GSM:',len(sample_cols),'/年齢取得:',len(selected_gsms))
    targets=read_gene_mapping(RNA_SEQ_GENE_MAP_CSV,gene_id)
    if targets is None:targets={str(gene_id)}
    values_by_gsm = {}
    libsize = pd.Series(0.,index=list(sample_cols))
    n_target_rows=0
    count_rows=0
    for chunk in pd.read_csv(matrix_path,sep=sep,usecols=[feature_col,*sample_cols.values()],
                             dtype={feature_col:str},chunksize=2000,encoding='utf-8-sig'):
        count_rows+=len(chunk)
        vals=chunk[list(sample_cols.values())].apply(pd.to_numeric,errors='coerce')
        bad = ~np.isfinite(vals.to_numpy(dtype=float))
        if mode == 'raw':
            if bad.any() or (vals.to_numpy(dtype=float)<0).any():
                raise ValueError('raw countsに非数値・欠損・負値があります。行列形式を確認してください。')
            libsize+=pd.Series(vals.sum(axis=0).to_numpy(dtype=float),index=list(sample_cols))
        hit=chunk[feature_col].astype(str).str.strip().isin(targets)
        if hit.any():
            n_target_rows+=int(hit.sum())
            for _, row in vals.loc[hit].iterrows():
                for gsm_id, col_name in sample_cols.items():
                    values_by_gsm.setdefault(gsm_id,[]).append(float(row[col_name]))
    if n_target_rows == 0:
        raise ValueError(f'Gene ID {gene_id}（対応features={sorted(targets)[:6]}）がRNA-seq行列のfeature列にありません。'+
                         ' Ensembl等のIDならRNA_SEQ_GENE_MAP_CSVを設定してください。')
    if n_target_rows > 1:
        raise ValueError(f'Gene ID {gene_id} に対応する行が{n_target_rows}件です。'+
                         ' transcript/重複featureの集約規則を人が指定する必要があります。自動で合算しません。')
    if mode == 'raw' and (libsize<=0).any():
        raise ValueError('raw countsのライブラリサイズが0の試料が存在します')
    out=[]
    for gsm_id in selected_gsms:
        if gsm_id not in values_by_gsm:continue
        raw=values_by_gsm[gsm_id][0]
        if not np.isfinite(raw) or raw<0:
            continue
        if mode=='raw':
            value=raw/libsize[gsm_id]*1_000_000
            metric='CPM'
        elif mode=='tpm':
            value=raw
            metric='TPM'
        else:
            value=raw
            metric='log2_preprocessed'
        if mode=='log2':
            transformed=value
        else:
            transformed=float(np.log2(value+RNA_SEQ_PSEUDOCOUNT))
        out.append(dict(gsm_id=gsm_id,probe_id=str(gene_id),feature_id=next(iter(targets)),
                        untransformed_value=float(raw),normalized_value=float(value),
                        expression=float(transformed),library_size=float(libsize[gsm_id]) if mode=='raw' else np.nan))
    return pd.DataFrame(out),{'n_matrix_rows':count_rows,'n_matched_features':n_target_rows,'n_samples_matrix':len(sample_cols),
                              'n_samples_used':len(out),'unmatched_gsms':sorted(missing)}

def get_rnaseq_expression():
    if RNA_SEQ_SOURCE.startswith('ncbi_'):
        path, provenance=download_ncbi_rnaseq(GSE_ID,RNA_SEQ_SOURCE)
        mode='raw' if RNA_SEQ_SOURCE=='ncbi_raw' else 'tpm'
    else:
        if not RNA_SEQ_MATRIX_PATH:raise ValueError('local_*の利用にはRNA_SEQ_MATRIX_PATHが必要です')
        path=Path(RNA_SEQ_MATRIX_PATH)
        provenance=str(path.resolve())
        mode={'local_raw':'raw','local_tpm':'tpm','local_log2':'log2'}[RNA_SEQ_SOURCE]
    df, audit=extract_rnaseq_target(path,GENE_ID,mode,metadata.gsm_id.tolist())
    if df.empty:raise ValueError('選択されたGSMに対象遺伝子のRNA-seq発現値がありません')
    meta=metadata[['gsm_id','platform_id','age','age_group']]
    df=df.merge(meta,on='gsm_id',validate='one_to_one',how='inner')
    df['gene_id']=GENE_ID
    df['data_type']='rna_seq'
    audit.update({'data_source':RNA_SEQ_SOURCE,'data_path':str(path),'provenance':provenance,
                  'metric':'log2(CPM+pseudocount)' if mode=='raw' else
                           'log2(TPM+pseudocount)' if mode=='tpm' else 'user_supplied_log2'})
    (root/'rna_seq_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    print('RNA-seqデータ:',audit['metric'], '\nソース:',path)
    return df, audit['metric']

## 6. データ抽出（技術別）

データ技術ごとに解析し、`expression.csv`に試料ごとの発現量を出力します。RNA-seqでは **Gene IDレベルで1行** として扱い、異なるGPLの値を統合しません（NCBIが作成した同一シリーズ内の行列については対応GSMを利用）。

In [ ]:
def get_microarray_expression(mapping):
    def extract_expression(gsm,probe_ids):
        table=gsm.table
        if not {'ID_REF','VALUE'}.issubset(table.columns):
            return None,'GSM_TABLE_MISSING_ID_REF_OR_VALUE'
        sub=table.loc[table['ID_REF'].astype(str).isin(probe_ids),['ID_REF','VALUE']].copy()
        sub['ID_REF']=sub['ID_REF'].astype(str)
        if sub.ID_REF.duplicated().any():return None,'DUPLICATE_PROBE_ID'
        sub['VALUE']=pd.to_numeric(sub.VALUE,errors='coerce')
        return sub.set_index('ID_REF')['VALUE'],'OK'
    
    expression=[]; issues=[]
    for gpl_id, mapping_sub in mapping.groupby('platform_id'):
        ids=set(mapping_sub.probe_id)
        for item in metadata.loc[metadata.platform_id==gpl_id].itertuples(index=False):
            gsm=gse.gsms[item.gsm_id]
            data,status=extract_expression(gsm,ids)
            if status!='OK':
                issues.append(dict(gsm_id=item.gsm_id,platform_id=gpl_id,status=status));continue
            for probe in ids:
                v=data.get(probe,np.nan)
                if pd.notna(v) and np.isfinite(v):
                    expression.append(dict(gsm_id=item.gsm_id,platform_id=gpl_id,probe_id=probe,
                        gene_id=GENE_ID,age=item.age,age_group=item.age_group,expression=float(v)))
                else:issues.append(dict(gsm_id=item.gsm_id,platform_id=gpl_id,status=f'MISSING_EXPRESSION:{probe}'))
    issues=pd.DataFrame(issues,columns=['gsm_id','platform_id','status'])
    issues.to_csv(root/'expression_issues.csv',index=False,encoding='utf-8-sig')
    if not expression:raise ValueError('解析可能なGSM発現値がありません。supplementaryファイルからの行列作成が必要な可能性があります')
    all_expression=pd.DataFrame(expression)
    all_expression.to_csv(root/'expression.csv',index=False,encoding='utf-8-sig')
    return all_expression

if assay == 'microarray':
    mapping=build_microarray_mapping()
    all_expression=get_microarray_expression(mapping)
    all_expression['data_type']='microarray'
    analysis_scale=EXPRESSION_SCALE
else:
    all_expression,analysis_scale=get_rnaseq_expression()
    all_expression.to_csv(root/'expression.csv',index=False,encoding='utf-8-sig')

print('対象発現値:',len(all_expression),'rows / 有効GSM:',all_expression.gsm_id.nunique())
display(all_expression.head(10))

## 7. 連続年齢・順位相関・若年/高齢の比較
- マイクロアレイ（log2確認時のみ）：元尺度の `2^(Old-Young)` をFold Changeとして計算。
- RNA-seq：log2(TPM+1)またはlog2(CPM+1)上の平均差を報告。**平均差の2のべき乗を通常のFold Changeと表記しません**（擬似カウント依存のため）。
- BH-FDRは「このNotebookで実際に検定したfeature集合」に対してのみ実施します。1遺伝子指定では全遺伝子探索の多重比較補正にはなりません。

In [ ]:
def analyze_probe(df):
    df=df.dropna(subset=['age','expression']).copy()
    row=dict(n=len(df),young_n=int((df.age_group=='Young').sum()),old_n=int((df.age_group=='Old').sum()),
        mean_expression=df.expression.mean())
    if len(df)>=MIN_REGRESSION_N and df.age.nunique()>=3 and df.expression.nunique()>=2:
        x=sm.add_constant(df['age'],has_constant='add')
        fit=sm.OLS(df.expression,x).fit()
        ci=fit.conf_int().loc['age']
        row.update(age_slope=fit.params['age'],age_slope_ci_low=ci.iloc[0],age_slope_ci_high=ci.iloc[1],age_p=fit.pvalues['age'])
        rho,sp=spearmanr(df.age,df.expression)
        row.update(spearman_rho=rho,spearman_p=sp)
    else:
        row.update(age_slope=np.nan,age_slope_ci_low=np.nan,age_slope_ci_high=np.nan,age_p=np.nan,spearman_rho=np.nan,spearman_p=np.nan)
    young=df.loc[df.age_group=='Young','expression'];old=df.loc[df.age_group=='Old','expression']
    row.update(young_mean=young.mean() if len(young) else np.nan,old_mean=old.mean() if len(old) else np.nan)
    row.update(old_minus_young=np.nan,difference_ci_low=np.nan,difference_ci_high=np.nan,welch_p=np.nan)
    if len(young)>=MIN_GROUP_N and len(old)>=MIN_GROUP_N:
        if young.var(ddof=1)>0 or old.var(ddof=1)>0:
            test=ttest_ind(old,young,equal_var=False)
            ci=test.confidence_interval(confidence_level=.95)
            row.update(old_minus_young=old.mean()-young.mean(),difference_ci_low=ci.low,difference_ci_high=ci.high,welch_p=test.pvalue)
    if assay=='microarray' and EXPRESSION_SCALE=='log2':
        row['old_minus_young_log2']=row['old_minus_young']
        for src,dst in [('old_minus_young','fold_change_old_vs_young'),('difference_ci_low','fold_change_ci_low'),('difference_ci_high','fold_change_ci_high')]:
            row[dst]=float(np.exp2(row[src])) if pd.notna(row[src]) else np.nan
    else:
        row['old_minus_young_log2']=np.nan
        for dst in ('fold_change_old_vs_young','fold_change_ci_low','fold_change_ci_high'):row[dst]=np.nan
    return row

summary_rows=[]
groupby_cols = ['platform_id','probe_id'] if assay=='microarray' else ['probe_id']
for key,df in all_expression.groupby(groupby_cols):
    if assay=='microarray':
        gpl,probe=key
    else:
        gpl,probe='RNA_SEQ',key[0]
    summary_rows.append(dict(gse_id=GSE_ID,gene_id=GENE_ID,data_type=assay,scale=analysis_scale,
                             platform_id=gpl,probe_id=probe,**analyze_probe(df)))
summary=pd.DataFrame(summary_rows)
for pcol, qcol in [('age_p','age_fdr_bh'),('spearman_p','spearman_fdr_bh'),('welch_p','welch_fdr_bh')]:
    summary[qcol]=np.nan
    eligible=summary[pcol].notna()
    if eligible.any():summary.loc[eligible,qcol]=multipletests(summary.loc[eligible,pcol],method='fdr_bh')[1]
summary['significant_at_0_05']=summary.welch_p.lt(.05) & summary.welch_p.notna()
summary['significance']=summary.welch_p.map(lambda p: 'NA' if pd.isna(p) else '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'ns')
summary.to_csv(root/'summary.csv',index=False,encoding='utf-8-sig')
display(summary)
print('Note: BH-FDRはこの実行で解析したプローブ集合内でのみ補正しています。')

## 8. プロットと解析条件の保存
RNA-seqはlog2(TPM+1) / log2(CPM+1)の図を作成します。遺伝子ごとのプローブ群の箱ひげ図はマイクロアレイのみ複数出ます。

In [ ]:
from matplotlib.ticker import MaxNLocator
plot_columns = ['platform_id','probe_id'] if assay=='microarray' else ['probe_id']
for k, df in all_expression.groupby(plot_columns):
    if assay=='microarray':gpl,probe=k
    else:gpl,probe='RNA_SEQ',k[0]
    identifier=re.sub(r'[^A-Za-z0-9_.-]+','_',f'{gpl}_{probe}')
    if len(df)>=MIN_REGRESSION_N and df.age.nunique()>=3 and df.expression.nunique()>=2:
        fit=sm.OLS(df.expression,sm.add_constant(df.age,has_constant='add')).fit()
        x=np.linspace(df.age.min(),df.age.max(),100)
        pred=fit.get_prediction(sm.add_constant(pd.Series(x,name='age'),has_constant='add')).summary_frame()
        fig,ax=plt.subplots(figsize=(7,4.5))
        ax.scatter(df.age,df.expression,alpha=.65)
        ax.plot(x,pred['mean'].to_numpy())
        ax.fill_between(x,pred['mean_ci_lower'].to_numpy(),pred['mean_ci_upper'].to_numpy(),alpha=.18)
        ax.set(title=f'{GSE_ID} | {gpl} | Gene {GENE_ID} | Probe {probe}',xlabel='Age (years)',ylabel=f'Expression ({analysis_scale})')
        fig.tight_layout();fig.savefig(root/f'{identifier}_age.png',dpi=160)
        if SHOW_PLOTS:plt.show()
        plt.close(fig)
    young=df.loc[df.age_group=='Young','expression'];old=df.loc[df.age_group=='Old','expression']
    if len(young)>=MIN_GROUP_N and len(old)>=MIN_GROUP_N:
        fig,ax=plt.subplots(figsize=(6,4.5))
        ax.boxplot([young,old],tick_labels=[f'Young (n={len(young)})',f'Old (n={len(old)})'])
        rng=np.random.default_rng(42)
        for i,grp in enumerate([young,old],1):ax.scatter(i+rng.uniform(-.08,.08,len(grp)),grp,alpha=.55,s=20)
        record=summary.loc[(summary.platform_id==gpl)&(summary.probe_id==probe)].iloc[0]
        ax.set(title=f'{GSE_ID} | {gpl} | Gene {GENE_ID} | Probe {probe}\nWelch p={record.welch_p:.3g}',ylabel=f'Expression ({analysis_scale})')
        fig.tight_layout();fig.savefig(root/f'{identifier}_groups.png',dpi=160)
        if SHOW_PLOTS:plt.show()
        plt.close(fig)
settings={k:globals()[k] for k in ['GSE_ID','GENE_ID','YOUNG_MAX','OLD_MIN','analysis_scale','AGE_OVERRIDE_CSV','CHARACTERISTIC_FILTERS','DROP_MISSING_AGE','MIN_REGRESSION_N','MIN_GROUP_N','GENE_COLUMN_OVERRIDES','INCLUDE_GSM_IDS','EXCLUDE_GSM_IDS','INCLUDE_GPL_IDS','TECHNOLOGY','RNA_SEQ_SOURCE','RNA_SEQ_MATRIX_PATH','RNA_SEQ_FEATURE_COLUMN','RNA_SEQ_GENE_MAP_CSV','RNA_SEQ_SAMPLE_MAP_CSV','RNA_SEQ_SEPARATOR','RNA_SEQ_PSEUDOCOUNT','RNA_SEQ_MAX_DOWNLOAD_MB']}
(root/'analysis_config.json').write_text(json.dumps(settings,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('保存先:',root.resolve())

## 9. 適用範囲・確認事項

1. **サンプル条件**：GEOの原著を確認し、加齢以外の要因（疾患・部位・細胞種・性別・バッチなど）の混入を吟味してください。年齢表記が「20-30歳」のような範囲なら自動推定しません。`AGE_OVERRIDE_CSV`を使用できます。
2. **GEO提供RNA-seq**：ヒトGRCh38.p13のNCBI生成Gene ID行列のみ自動取得します。GEOに行列が存在しない場合は、`local_*`を用いて配布元の行列を指定します。RNA-seq raw FASTQ/SRA の処理は行いません。
3. **解析法**：`ncbi_tpm`はlog2(TPM+1)。`ncbi_raw`は**全行に対するライブラリサイズ**でCPM化しlog2(CPM+1)。これらは探索的な発現量比較のための近似です。厳密なRNA-seq差次的発現解析には適切なカウントモデル・サイズファクター調整等が必要です。
4. **Gene ID**：NCBI生成行列はNCBI Gene ID。独自行列でEnsemblやGene Symbolしかない場合は `RNA_SEQ_GENE_MAP_CSV`（feature_id,gene_id）を明示してください。複数featureがGene IDに対応する際は勝手に集約しません。
5. **Fold Change**：マイクロアレイでログ2発現量の根拠がある場合のみ計算。RNA-seqのlog2(TPM+1)等の群平均差からは通常のFCを計算しません。
6. **繰返しサンプル**：同一人の反復測定、複数バッチ、群の偏りには本Notebookの単純なOLSやWelchは適しません。モデル拡張が必要です。
7. **検証状態**：合成行列を用いたローカルテスト、Notebookの構文テストを実施できますが、実際のGEOに対する一気通貫のダウンロード・実行はネットワーク環境で別途検証が必要です。

**根拠**： [NCBI生成RNA-seq count data](https://www.ncbi.nlm.nih.gov/geo/info/rnaseqcounts.html) ・ [NCBI GEO Programmatic access](https://www.ncbi.nlm.nih.gov/geo/info/geo_paccess.html)。
8. **GSE113957について**：健常者のみを用いる場合は `CHARACTERISTIC_FILTERS={"disease":["Normal"]}` としてください。HGPSサンプルは通常加齢の参照集団とは別に検討します。公開されている投稿者提供行列はFPKMです。NCBI由来TPM自動取得が必ず利用できるという意味ではありません。
